# Demo 5：Electrochemical Conversion

任务：`electrochemical-conversion`。本 Demo 展示 probe—diagnose—adapt—control 闭环：先用 pH/UV-vis 反馈诊断，再改变电势和电流，观察选择性、法拉第效率、传递和欧姆损失。

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'chemworld').exists()), None)
if ROOT is None:
    raise RuntimeError('请从 ChemWorld 仓库内启动 notebook')
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
du = importlib.import_module('notebooks.task_demos.demo_utils')

TASK_ID = 'electrochemical-conversion'
SEED = 0

## 1. 公开任务合同

公开合同允许设定电解质 profile、电势、电流与时间，并返回诊断反馈。Demo 不读取 hidden state、交换电流、传递系数或真实过电势。

In [2]:
card = du.task_card(TASK_ID)
pd.DataFrame({
    'field': ['task_id', 'motivation', 'budget', 'episode_mode', 'success_metrics'],
    'value': [card['task_id'], card['scientific_motivation'], card['budget'], card['episode_mode'], card['reward_leaderboard_metric']['success_metrics']],
})

,field,value
0,task_id,electrochemical-conversion
1,motivation,"Select a bounded aqueous electrolyte profile, ..."
2,budget,48
3,episode_mode,campaign
4,success_metrics,"[score, electrochemical_selectivity, faradaic_..."


## 2. 构造候选干预

每个候选 recipe 都先执行诊断 probe，再根据预设候选改变 setpoint。真正的 Agent 可以用第一次反馈动态生成第二个 setpoint。

In [3]:
vectors = du.standard_probe_vectors(TASK_ID)
mid_recipe = du.recipe_frame(TASK_ID, vectors['mid'])
mid_recipe

,step,operation,payload
0,1,add_solvent,"{'operation': 'add_solvent', 'volume_L': 0.025..."
1,2,add_reagent,"{'operation': 'add_reagent', 'amount_mol': 0.0..."
2,3,set_potential,"{'operation': 'set_potential', 'potential_V': ..."
3,4,electrolyze,"{'operation': 'electrolyze', 'duration_s': 540.0}"
4,5,measure,"{'operation': 'measure', 'instrument': 'ph_met..."
5,6,measure,"{'operation': 'measure', 'instrument': 'uvvis'}"
6,7,set_potential,"{'operation': 'set_potential', 'potential_V': ..."
7,8,electrolyze,"{'operation': 'electrolyze', 'duration_s': 195..."
8,9,measure,"{'operation': 'measure', 'instrument': 'uvvis'}"
9,10,terminate,{'operation': 'terminate'}


## 3. 执行并读取反馈

比较公开 `processed_estimate` 中的选择性、法拉第效率、能量效率、传递效率和 pH/沉淀信号。

In [4]:
comparison = du.compare_vectors(TASK_ID, vectors, seed=SEED)
columns = ['candidate', 'leaderboard_score', 'electrochemical_selectivity', 'faradaic_efficiency', 'transport_efficiency', 'ohmic_efficiency', 'energy_efficiency', 'pH_normalized', 'all_actions_valid']
comparison[[column for column in columns if column in comparison]]

,candidate,leaderboard_score,electrochemical_selectivity,faradaic_efficiency,transport_efficiency,ohmic_efficiency,energy_efficiency,pH_normalized,all_actions_valid
0,low,0.593897,0.916151,0.553940,0.568378,0.868476,0.635870,0.234157,True
1,mid,0.747597,0.916151,0.895500,0.999938,0.977078,0.741885,0.157448,True
2,high,0.450212,0.883163,0.302468,0.316905,0.911612,0.205904,0.300535,True


### 查看诊断前后反馈

测量时间顺序用于判断第二个 setpoint 是否真正依赖反馈。

In [5]:
run = du.run_vector(TASK_ID, vectors['mid'], seed=SEED)
measurement_feedback = du.measurement_trace(run)
measurement_feedback

,step,operation,reward,leaderboard_score,observed_keys,processed_estimate
0,5,measure,0.000000,NaN,"cost, safety_risk, score, pH_normalized, preci...","{'pH_normalized': 0.15742198082726305, 'precip..."
1,6,measure,0.479831,NaN,"yield, conversion, energy_efficiency, faradaic...","{'yield': 0.06470240646129677, 'conversion': 0..."
2,9,measure,0.024093,NaN,"yield, conversion, energy_efficiency, faradaic...","{'yield': 0.0025049294499239066, 'conversion':..."
3,11,measure,0.243673,0.747597,"yield, selectivity, conversion, byproduct_sign...","{'yield': 0.05997345750198079, 'selectivity': ..."


## 4. 同一干预，不同隐藏规律

World B 使用 `constitutive_law_family` 教学控制改变隐藏电荷转移、选择性衰减和标准电势关系；公开接口保持不变。

In [6]:
paired_worlds = du.compare_hidden_worlds(
    TASK_ID, vectors['mid'], mechanism_mode='constitutive_law_family', seed=SEED
)
paired_worlds

,opaque_world,score,electrochemical_selectivity,faradaic_efficiency,transport_efficiency,ohmic_efficiency,energy_efficiency,pH_normalized,precipitation_signal,safety_risk,leaderboard_score,cost,all_actions_valid
0,World A,None,0.916151,0.8955,0.999938,0.977078,0.741885,0.157448,0.008748,None,0.747597,0.0,True
1,World B,None,0.916151,0.8955,0.999938,0.977078,0.884781,0.157448,0.008748,None,0.768125,0.0,True


## 5. 留给 Agent 的能力问题

Agent 应利用诊断反馈更新当前 world model，判断限制来自电荷转移、传质、欧姆损失还是副反应，而不是只追逐更高电势。

In [7]:
capability_probe = {
    'current_evidence': comparison.to_dict(orient='records'),
    'prediction_query': '预测一个未执行 potential/current 条件的选择性与效率分布。',
    'next_intervention_query': '选择最能区分动力学、传递和欧姆限制的下一次 probe。',
    'evaluation_note': '检验反馈是否改变预测和 setpoint，而不是只看操作次数。',
}
capability_probe

{'current_evidence': [{'candidate': 'low',
   'score': None,
   'electrochemical_selectivity': 0.9161505050853876,
   'faradaic_efficiency': 0.5539401158953059,
   'transport_efficiency': 0.5683775816190109,
   'ohmic_efficiency': 0.8684755253078726,
   'energy_efficiency': 0.6358704995638846,
   'pH_normalized': 0.2341565464606179,
   'precipitation_signal': 0.6925203581015684,
   'safety_risk': None,
   'leaderboard_score': 0.5938968862399511,
   'cost': 0.0,
   'all_actions_valid': True,
   'operation_count': 11},
  {'candidate': 'mid',
   'score': None,
   'electrochemical_selectivity': 0.9161505050853876,
   'faradaic_efficiency': 0.895500176412614,
   'transport_efficiency': 0.9999376421363183,
   'ohmic_efficiency': 0.977077699220916,
   'energy_efficiency': 0.7418851688887946,
   'pH_normalized': 0.15744797120354811,
   'precipitation_signal': 0.008748124101221753,
   'safety_risk': None,
   'leaderboard_score': 0.7475973344272242,
   'cost': 0.0,
   'all_actions_valid': True,
